# Script to Calibrate the Recovery rate

In [ ]:
from pathlib import Path

import numpy as np 
import plotly.graph_objects as go

from pcr.model import PCRModel
from pcr import builder

In [ ]:
# model configuration 
date_start = '2000'
date_end = '2099'

# initialize model 
model = PCRModel(
    year_start=date_start,
    year_end=date_end,
    nr_simulation=1000,
    nr_batch=1000,
    scenario='0',
    rec_rate= 7.57/365, # 7.57/365
    fac_lambda=np.array([[2015, 2100], [0.9, 0.9]])
)

# model.run()

# Calibrate rec_rate

Search rec_rate in [7/365, 8/365] to minimize abs(sum(median shoreline)) over the ensemble.

Wave loading / storm detection are rec_rate-independent, so they run once via `init_slr`, `load_wave_data`, `detect_storms`. Each candidate rec_rate then only re-runs `run_simulation()`. The RNG is seeded per-evaluation so the objective is deterministic in rec_rate (otherwise storm resampling noise would make the search unstable).

In [ ]:
# build slr and wave input 
rate_slr, days_slr = builder.build_slr_curve(scenario='0', transect=None, date_start=model.date_start)
model.attach_slr(rate_slr, days_slr)

wave_data_path= '../data/ERA5/B3_offshore.nc'
hs, dir, tp, day, record_years = builder.build_wave_array('file', wave_data_path=wave_data_path)
model.attach_wave_data(hs, dir, tp, day, record_years)

# detect storm only once 
model.detect_storms()

In [ ]:
grid_days = np.arange(1, 366*100, 30)
SEED = 0

def sum_abs_med(median_all):
    return np.sum(np.abs(median_all))

def evaluate_rec_rate(rec_rate):
    model.rec_rate = rec_rate

    np.random.seed(SEED)  # same storm sequence for every candidate rec_rate
    model.run_simulation()

    aligned = np.empty((len(grid_days), len(model.track_time)))
    for i, (t, s) in enumerate(zip(model.track_time, model.track_shoreline)):
        aligned[:, i] = np.interp(grid_days, t, s)

    median_all = np.median(aligned, axis=1)
    return sum_abs_med(median_all)

In [ ]:
from scipy.optimize import minimize_scalar

result = minimize_scalar(
    evaluate_rec_rate,
    bounds=(6/365, 7/365),
    method='bounded',
    options={'xatol': 1e-6},
)

print(f'best rec_rate: {result.x:.6f}  (x*365 = {result.x*365:.4f} m/year)')
print(f'objective: {result.fun:.4f}')

In [ ]:
# optional: sanity-check the objective shape over the search range
candidates = np.linspace(6/365, 7.5/365, 11)
objectives = [evaluate_rec_rate(r) for r in candidates]

for r, obj in zip(candidates, objectives):
    print(f'rec_rate={r:.6f} ({r*365:.3f} m/yr) -> objective={obj:.4f}')

In [ ]:
# sanity check see if the slr 0 
rate, days = model.init_slr()

from pcr import slr 

synth_slr = slr.vector_simulate_slrAR6(
            day_start=model.track_time[0],
            rate_sl=rate,
            day_sl=days,
            wl0=model.wl0,
            scenario=model.ar6_scenario,
        )

print(np.max(np.abs(synth_slr)))